In [1]:
import sys
from pathlib import Path

# Add src/ to Python path so we can import cv_strategies
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
from src.cv_strategies import (
    random_kfold,
    temporal_holdout,
    temporal_blocked_kfold,
    leave_one_site_out,
)

print("All imports OK")

All imports OK


In [2]:
# Load the cleaned metadata (we don't need OTU tables for testing the splits)
meta = pd.read_csv("../data/processed/meta_clean.csv", index_col=0)
print(f"meta: {meta.shape}\n")

# Test target: drop missing temperature values (so we test on a realistic subset)
valid_mask = meta['temperature'].notna()
meta_valid = meta.loc[valid_mask].reset_index(drop=True)
n = len(meta_valid)
print(f"Valid samples for 'temperature': {n}")
print(f"Date range: {pd.to_datetime(meta_valid['date']).min().date()}  →  "
      f"{pd.to_datetime(meta_valid['date']).max().date()}")
print(f"Number of sites: {meta_valid['location_name'].nunique()}\n")

# ---- Strategy 1: random_kfold ----
print("=" * 60)
print("1. random_kfold (n_folds=5)")
print("=" * 60)
splits = random_kfold(n_samples=n, n_folds=5, random_seed=42)
for i, (tr, te) in enumerate(splits, 1):
    print(f"  Fold {i}: train={len(tr):4d}, test={len(te):3d}")

# ---- Strategy 2: temporal_holdout ----
print("\n" + "=" * 60)
print("2. temporal_holdout (test_fraction=0.2)")
print("=" * 60)
splits = temporal_holdout(meta_valid, test_fraction=0.2)
tr, te = splits[0]
print(f"  Single split: train={len(tr)}, test={len(te)}")
tr_dates = pd.to_datetime(meta_valid.iloc[tr]['date'])
te_dates = pd.to_datetime(meta_valid.iloc[te]['date'])
print(f"  Train dates: {tr_dates.min().date()}  →  {tr_dates.max().date()}")
print(f"  Test  dates: {te_dates.min().date()}  →  {te_dates.max().date()}")

# ---- Strategy 3: temporal_blocked_kfold ----
print("\n" + "=" * 60)
print("3. temporal_blocked_kfold (n_folds=5)")
print("=" * 60)
splits = temporal_blocked_kfold(meta_valid, n_folds=5)
for i, (tr, te) in enumerate(splits, 1):
    te_dates = pd.to_datetime(meta_valid.iloc[te]['date'])
    print(f"  Fold {i}: train={len(tr):4d}, test={len(te):3d}  "
          f"test range: {te_dates.min().date()}  →  {te_dates.max().date()}")

# ---- Strategy 4: leave_one_site_out ----
print("\n" + "=" * 60)
print("4. leave_one_site_out")
print("=" * 60)
splits = leave_one_site_out(meta_valid)
print(f"  Total folds: {len(splits)}  (one per site)")
for i, (tr, te) in enumerate(splits, 1):
    site = meta_valid.iloc[te[0]]['location_name']
    print(f"  Fold {i:2d} ({site:30s}): train={len(tr):4d}, test={len(te):3d}")

meta: (1538, 24)

Valid samples for 'temperature': 1223
Date range: 2022-04-25  →  2023-04-27
Number of sites: 14

1. random_kfold (n_folds=5)
  Fold 1: train= 978, test=245
  Fold 2: train= 978, test=245
  Fold 3: train= 978, test=245
  Fold 4: train= 979, test=244
  Fold 5: train= 979, test=244

2. temporal_holdout (test_fraction=0.2)
  Single split: train=978, test=245
  Train dates: 2022-04-25  →  2023-02-23
  Test  dates: 2023-02-23  →  2023-04-27

3. temporal_blocked_kfold (n_folds=5)
  Fold 1: train= 978, test=245  test range: 2022-04-25  →  2022-08-11
  Fold 2: train= 978, test=245  test range: 2022-08-11  →  2022-10-17
  Fold 3: train= 978, test=245  test range: 2022-10-17  →  2022-12-26
  Fold 4: train= 979, test=244  test range: 2022-12-26  →  2023-02-23
  Fold 5: train= 979, test=244  test range: 2023-02-23  →  2023-04-27

4. leave_one_site_out
  Total folds: 14  (one per site)
  Fold  1 (Boergerende Strand            ): train=1136, test= 87
  Fold  2 (Diedrichshagen       